In [16]:
!pip install requests python-dotenv
# requests let python download data from websites
# .env lets python secretely read api_key in .env

In [22]:
#test_connection.py
#basic import
import os
import requests
from dotenv import load_dotenv

#load secret api_key
load_dotenv()
api_key = os.getenv("TMDB_API_KEY")

print("API Key:", api_key)
#test for movie=inception
movie_id = 27205
url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}"
print(url)
# get the response from the url
print("Connecting to TMDB...")
response = requests.get(url, timeout=30)

# 4. Check if it worked!
if response.status_code == 200:
    data = response.json()# important data = response=requests.url(f"https/{}/{}").json()
    print("\n✅ Success! Connection established.")
    print(f"Movie Title: {data['title']}")
    print(f"Tagline: {data['tagline']}")
    print(f"Poster Path: {data['poster_path']}")
else:
    print(f"\n❌ Failed to connect. Error code: {response.status_code}")
    print(response.text)

API Key: 1735e9c7ea3db1ace652026c247e16b3
https://api.themoviedb.org/3/movie/27205?api_key=1735e9c7ea3db1ace652026c247e16b3
Connecting to TMDB...


ConnectTimeout: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/27205?api_key=1735e9c7ea3db1ace652026c247e16b3 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001E36616D410>, 'Connection to api.themoviedb.org timed out. (connect timeout=30)'))

In [18]:
import os
import time
import json
import requests
from dotenv import load_dotenv

# 1. Setup paths based on our new folder structure
load_dotenv()
API_KEY = os.getenv("TMDB_API_KEY")

# Get the absolute path to the root of our project
#BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), "../../"))
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_FILE = os.path.join(DATA_DIR, "movies_raw.jsonl")

def fetch_movies(pages_to_fetch=2):
    print(f"Starting data pipeline. Saving to: {OUTPUT_FILE}")

    # Open the file in 'append' mode so we don't overwrite data if it crashes
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:

        for page in range(1, pages_to_fetch + 1):
            print(f"Fetching page {page}...")

            url = f"https://api.themoviedb.org/3/discover/movie?api_key={API_KEY}&sort_by=popularity.desc&page={page}&language=en-US"
            response = requests.get(url)

            if response.status_code == 200:
                data = response.json()

                # Loop through each movie on this page
                for movie in data["results"]:
                    # We only save the fields we actually need for our ML models
                    movie_payload = {
                        "id": movie.get("id"),
                        "title": movie.get("title"),
                        "overview": movie.get("overview"), # For NLP (Emotions)
                        "genre_ids": movie.get("genre_ids"),
                        "poster_path": movie.get("poster_path"), # For Computer Vision
                        "release_date": movie.get("release_date")
                    }

                    # Write it as a JSON line
                    f.write(json.dumps(movie_payload) + "\n")

                # Be polite to the API, wait a quarter of a second before the next page
                time.sleep(0.25)

            else:
                print(f"Error on page {page}: {response.status_code}")
                break

    print("Pipeline finished successfully! Check your data folder.")

if __name__ == "__main__":
    fetch_movies(pages_to_fetch=500)


Starting data pipeline. Saving to: c:\Users\HP\Downloads\cinema-mind-recommender\cinema-mind-recommender\data\movies_raw.jsonl
Fetching page 1...


ConnectTimeout: HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/discover/movie?api_key=1735e9c7ea3db1ace652026c247e16b3&sort_by=popularity.desc&page=1&language=en-US (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001E366193A50>, 'Connection to api.themoviedb.org timed out. (connect timeout=None)'))

In [ ]:
!python /content/drive/MyDrive/cinema-mind-recommender/src/data_pipeline/download_posters.py

Reading data from: /content/drive/MyDrive/cinema-mind-recommender/data/movies_raw.jsonl
Saving images to: /content/drive/MyDrive/cinema-mind-recommender/data/posters
Finished downloading posters!


In [6]:
!pip install pandas sentence-transformers torch

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 4.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ------- -------------------------------- 2.1/11.0 MB 9.0 MB/s eta 0:00:01
   ------------ --------------------------- 3.4/11.0 MB 8.4 MB/s eta 0:00:01
   --------------------- ------------------ 6.0/11.0 MB 9.2 MB/s eta 0:00:01
   ------------------------------- -------- 8.7/11.0 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 10.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/684.4 kB ? eta -:--:--
   ---------------------------------------- 684.4/684.4 kB 6.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -------------------------- ------------- 2.6/4.0 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 12.7 MB/s eta 0:00:00
   --------------

In [9]:
!pip install pyarrow

In [10]:
!python /content/drive/MyDrive/cinema-mind-recommender/src/embedding_engine/text_embedder.py

Loading raw data from: /content/drive/MyDrive/cinema-mind-recommender/data/movies_raw.jsonl
Loading HuggingFace Transformer Model...
modules.json: 100% 349/349 [00:00<00:00, 1.12MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 310kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 16.9MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 190kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.13MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 70.2MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 18163.27it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.57MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 34.7MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 435kB/s]
config.json: 100% 190/190 [00:00<00:00, 731kB/s]
Converting 40 movies into math... This might take a minute.
Batches: 100% 2/2 [00:02<00:00,  1.09s/it]
Saving to highly compressed Parquet format at: /content/drive/MyDrive/cinema-mind-

In [8]:
!pip install torchvision pillow tqdm

In [ ]:
!python /content/drive/MyDrive/cinema-mind-recommender/src/vision_module/image_embedder.py

Loading Pre-trained ResNet-50 Model...
Found 40 posters to process.
Extracting Visual Features: 100% 40/40 [00:07<00:00,  5.41it/s]

Saving visual embeddings to: /content/drive/MyDrive/cinema-mind-recommender/data/visual_embeddings.parquet
✅ Visual embeddings generated successfully!


In [10]:
!pip install qdrant-client

   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------- ----------------------------- 1.3/4.9 MB 6.7 MB/s eta 0:00:01
   ----------------------- ---------------- 2.9/4.9 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 8.3 MB/s eta 0:00:00

   ---------------- ----------------------- 2/5 [grpcio]
   ---------------- ----------------------- 2/5 [grpcio]
   ---------------- ----------------------- 2/5 [grpcio]
   ------------------------ --------------- 3/5 [h2]
   -------------------------------- ------- 4/5 [qdrant-client]
   -------------------------------- ------- 4/5 [qdrant-client]
   -------------------------------- ------- 4/5 [qdrant-client]
   -------------------------------- ------- 4/5 [qdrant-client]
   -------------------------------- ------- 4/5 [qdrant-client]
   -------------------------------- ------- 4/5 [qdrant-client]
   ---------------------------------------- 5/5 [qdrant-client]



In [1]:
!python src/vector_db/build_index.py

Loading Parquet files...
Merged successfully. Preparing 9958 movies for the Vector DB.
Configuring Qdrant Collection with Named Vectors...
Beginning batched upsert routines (Chunk Size: 500)...
  Successfully processed points 0 through 500...
  Successfully processed points 500 through 1000...
  Successfully processed points 1000 through 1500...
  Successfully processed points 1500 through 2000...
  Successfully processed points 2000 through 2500...
  Successfully processed points 2500 through 3000...
  Successfully processed points 3000 through 3500...
  Successfully processed points 3500 through 4000...
  Successfully processed points 4000 through 4500...
  Successfully processed points 4500 through 5000...
  Successfully processed points 5000 through 5500...
  Successfully processed points 5500 through 6000...
  Successfully processed points 6000 through 6500...
  Successfully processed points 6500 through 7000...
  Successfully processed points 7000 through 7500...
  Successfully p

c:\Users\HP\Downloads\cinema-mind-recommender\cinema-mind-recommender\src\vector_db\build_index.py:37: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


In [4]:
!python src/backend/search.py


🧠 Analyzing your prompt: 'A psychological thriller with a great twist ending starring Leonardo DeCaprio'...
⚡ Searching the Vector Database...

🍿 HERE ARE YOUR RECOMMENDATIONS 🍿
🎬 The Exorcism (Match Score: 55.7%)
📝 A troubled actor begins to unravel while shooting a supernatural horror film, leading his estranged daughter to wonder if he's slipping back into his ...

🎬 NN4444 (Match Score: 53.8%)
📝 Four unsettling short films exploring psychological horror: a man who barks like a dog at strangers, madness at a party, eerie phenomena following a f...

🎬 Psycho (Match Score: 51.4%)
📝 Based on the Buddhist tale of Angulimala, a dreaded serial killer, Psycho tells the story of a blind man who gets involved in a murder mystery. trying...




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1920.43it/s]


In [3]:
!pip install streamlit

In [13]:
!pip uninstall peft -y

Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1


In [14]:
!streamlit run src\frontend\app.py

^C


In [15]:
!pip freeze > requirements.txt